In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

all_data = []

print(f"\nScraping category: MANUFACTURE...\n")

for page in range(1, 20):
    print(f"  Scraping page {page}...")

    url = f"https://ensun.io/search?threshold=VERY_LOW&q=UPVC%20pipe&page={page}&locations=United%20States%2Cnull%2Cnull&categories=MANUFACTURER"
    driver.get(url)
    time.sleep(3)  # Allow dynamic content to load

    listings = driver.find_elements(By.XPATH, '//div[contains(@class,"MuiPaper-root")]')

    if not listings:
        print("  No listings found, stopping.\n")
        break

    for listing in listings:
        try:
            company_name = listing.find_element(By.XPATH, './/p[contains(@class,"mui-vpiwmb")]').text.strip()
        except:
            company_name = "Not Found"

        try:
            location = listing.find_element(By.XPATH, './/p[contains(@class,"mui-opaoyh")]').text.strip()
        except:
            location = "Not Found"

        try:
            details = listing.find_elements(By.XPATH, './/p[contains(@class,"mui-1k5s1ae")]')
            employees = details[0].text.strip() if len(details) >= 1 else "Not Found"
            founded = details[1].text.strip() if len(details) >= 2 else "Not Found"
        except:
            employees = "Not Found"
            founded = "Not Found"

        try:
            key_descrip = listing.find_element(By.XPATH, './/p[contains(@class,"mui-1tjpl0f")]').text.strip()
        except:
            key_descrip = "Not Found"

        all_data.append({
            "Company Name": company_name,
            "Location": location,
            "Employee Count": employees,
            "Year Founded": founded,
            "Key takeaway": key_descrip
        })

print("\n✅ Finished scraping distributors.")
driver.quit()

# Save to Excel
df = pd.DataFrame(all_data).drop_duplicates()
df


Scraping category: MANUFACTURE...

  Scraping page 1...
  Scraping page 2...
  Scraping page 3...
  Scraping page 4...
  Scraping page 5...
  Scraping page 6...
  Scraping page 7...
  Scraping page 8...
  Scraping page 9...
  Scraping page 10...
  No listings found, stopping.


✅ Finished scraping distributors.


,Company Name,Location,Employee Count,Year Founded,Key takeaway
0,Not Found,Not Found,Not Found,Not Found,Not Found
3,PVC Pipe Supplies,"Olive Branch, United States",1-10 Employees,2015,PVC Pipe Supplies specializes in a variety of ...
4,U.S. Pipe,"Birmingham, United States",501-1000 Employees,1899,"U.S. Pipe, a Quikrete company, specializes in ..."
5,United Pipe & Steel,"Ipswich, United States",251-500 Employees,1920,United Pipe & Steel is a prominent master dist...
7,PVF Industrial Inc.,"Tampa, United States",11-50 Employees,2019,PVF Industrial is a reliable manufacturer and ...
...,...,...,...,...,...
130,Drinkwater Products,United States,11-50 Employees,1987,Drinkwater Products specializes in pipeline ma...
131,IMPREG LLC,"Richmond, United States",51-100 Employees,1999,IMPREG is a leading manufacturer of fiberglass...
132,Haviland Drainage Products Co.,"Haviland, United States",11-50 Employees,1924,Haviland Drainage Products Co. specializes in ...
133,Specified Fittings,"Bellingham, United States",11-50 Employees,1997,Specified Fittings is a prominent manufacturer...


In [2]:
df.to_excel("Ensun_UPVC_Manufacture_dipipe.xlsx", index=False)
print("\n✅ Data saved to 'Ensun_UPVC_Manufacture_dipipe.xlsx'")


✅ Data saved to 'Ensun_UPVC_Manufacture_dipipe.xlsx'


In [12]:
import pandas as pd
import re

# Load the Excel file
df = pd.ExcelFile('Ensun_UPVC_Manufacture_dipipe.xlsx')
data = df.parse('Sheet1')

# Function to identify product type and split location
def process_row(row):
    key_takeaway = str(row['Key takeaway']).lower()
    product_type = 'Yes' if re.search(r'\bupvc\b', key_takeaway) or 'upvc pipes' in key_takeaway else 'No'
    
    location = str(row['Location'])
    if ',' in location:
        city, state = [x.strip() for x in location.split(',', 1)]
    else:
        city, state = '', location.strip()
    
    return pd.Series([product_type, city, state])

# Apply processing
data[['Product Type', 'Location City', 'Location State']] = data.apply(process_row, axis=1)

# Remove the first row (assumed invalid)
data = data.iloc[1:].reset_index(drop=True)

# Clean 'Employee Count' to remove 'Employees'
data['Employee Count'] = data['Employee Count'].str.replace(' Employees', '', regex=False)

# Select required columns
output_data = data[[
    'Company Name',
    'Location City',
    'Location State',
    'Employee Count',
    'Year Founded',
    'Key takeaway',
    'Product Type'
]]

# Export the final output
output_data.to_excel('UPVC_ensun_COMPANY_DETAILS.xlsx', index=False)

print("Excel file with all employee counts saved as 'UPVC_ensun_COMPANY_DETAILS.xlsx'")


Excel file with all employee counts saved as 'UPVC_ensun_COMPANY_DETAILS.xlsx'
